
# ULIRG→QSO evolutionary sequence: dust-obscured starburst to bare quasar

Sanders et al. (1988) proposed that Ultra-Luminous Infrared Galaxies (ULIRGs)
are the dust-shrouded precursors to optical QSOs. This sequence traces
progressive unveiling of a buried AGN through five stages:

1. **Max-Obscured Starburst** (τ_V = 5.0): Dust-buried, negligible AGN.
2. **Dust-Rich Seyfert 2** (τ_V = 2.0): AGN emerges through dust; mixed SFR/AGN.
3. **Moderate Seyfert** (τ_V = 0.5): AGN brightens; dust thins; transition regime.
4. **Weak-Dust QSO** (τ_V = 0.05): AGN dominates; dust nearly cleared.
5. **Bare Type-1 QSO** (τ_V ≈ 0): Pure unobscured AGN; UV-to-FIR from hot disc.

As obscuration decreases, the rest-frame SED morphology transforms:
the FIR dust-emission bump shrinks while the UV continuum brightens.

**References:**

- Sanders et al. (1988) ApJ 325, 74: ULIRG/QSO connection hypothesis

- Veilleux et al. (2009) ARA&A 47, 63: ULIRG/QSO transition review


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

# Physical constant: c in Angstrom/s
C_AA_PER_S = 2.998e18

ssp = tengri.load_ssp()

# ============================================================================
# Define the ULIRG→QSO sequence: (tau_V, agn_lum_ratio) pairs
# ============================================================================
# tau_V governs the visual optical depth of dust attenuation (Calzetti law).
# agn_lum_ratio is the AGN luminosity fraction of the total bolometric output.
#
# Sequence progression:
#  (5.0, 0.00) → (2.0, 0.40) → (0.5, 0.70) → (0.05, 0.95) → (0.00, 1.00)

sequence = [
    {"stage": "Max-Obscured ULIRG", "tau_v": 5.0, "agn_lum_ratio": 0.00},
    {"stage": "Dust-Rich Sy2", "tau_v": 2.0, "agn_lum_ratio": 0.40},
    {"stage": "Moderate Seyfert", "tau_v": 0.5, "agn_lum_ratio": 0.70},
    {"stage": "Weak-Dust QSO", "tau_v": 0.05, "agn_lum_ratio": 0.95},
    {"stage": "Bare Type-1 QSO", "tau_v": 0.00, "agn_lum_ratio": 1.00},
]

# ============================================================================
# Helper: build a single archetype model
# ============================================================================


def build_ulirg_qso_model(tau_v, agn_lum_ratio):
    """
    Construct a SEDModel along the ULIRG→QSO sequence.

    Parameters
    ----------
    tau_v : float
        Visual optical depth (dust attenuation) [dimensionless].
    agn_lum_ratio : float
        AGN luminosity fraction [0–1].

    Returns
    -------
    model : tengri.SEDModel
        Fitted model with fixed SFH, dust, and AGN parameters.
    params : dict
        Sampled parameter dict.
    """
    # Fiducial bolometric AGN luminosity (log L_bol / L_sun)
    log_lbol = 12.0

    # SFR decreases as AGN fraction increases (energy budget constraint).
    # At agn_lum_ratio=0 (pure starburst): high SFR; at agn_lum_ratio=1 (pure QSO): low SFR.
    # ``sfh.type=const`` parametrizes by total stellar mass over the default
    # window [start_gyr=13.8, end_gyr=0 Gyr]; convert via M = SFR × Δt:
    #   log_total_mass = log_sfr + log10(1.38e10 yr) ≈ log_sfr + 10.14
    log_sfr = 1.5 - 2.0 * agn_lum_ratio
    log_total_mass = log_sfr + 10.14

    # Dust emission is computed from tau_v (via two-component reddening).
    # tau_diff and tau_bc scale with tau_v; here we use a simple proportionality.
    # tau_V = tau_bc + tau_diff (approximation), splitting ~80% birth cloud, ~20% diffuse.
    tau_bc = 0.8 * tau_v
    tau_diff = 0.2 * tau_v

    agn_dict = {
        "type": "composable",
        "log_lbol": log_lbol,
        "lum_ratio": agn_lum_ratio,
        "disc": {"type": "multicolor", "all_params": tengri.Fixed(tengri.DEFAULT)},
        "torus": {"type": "skirtor", "all_params": tengri.Fixed(tengri.DEFAULT)},
        "nlr": {"type": "analytic", "all_params": tengri.Fixed(tengri.DEFAULT)},
        "blr": {"type": "none", "all_params": tengri.Fixed(tengri.DEFAULT)},
        "all_params": tengri.Fixed(tengri.DEFAULT),
    }

    model = tengri.SEDModel.build(
        ssp,
        sfh={
            "type": "const",
            "all_params": tengri.Fixed(tengri.DEFAULT),
            "log_total_mass": log_total_mass,
        },
        dust_attenuation={
            "law": "power_law",
            "type": "two_component",
            "all_params": tengri.Fixed(tengri.DEFAULT),
            "tau_bc": tau_bc,
            "tau_diff": tau_diff,
        },
        dust_emission={"type": "dale2014", "all_params": tengri.Fixed(tengri.DEFAULT)},
        agn=agn_dict,
        redshift=tengri.Fixed(0.0),
    )

    params = dict(model.spec.sample(jax.random.PRNGKey(0)))
    return model, params


# ============================================================================
# Compute SEDs for all five archetypes
# ============================================================================

seds = []
for item in sequence:
    model, params = build_ulirg_qso_model(item["tau_v"], item["agn_lum_ratio"])
    out = model.predict(params)
    wave = np.asarray(model.wavelengths)
    sed = np.asarray(out.rest_sed())
    nu_l_nu = (C_AA_PER_S / wave) * sed
    seds.append({"stage": item["stage"], "wave": wave, "nu_l_nu": nu_l_nu})

# ============================================================================
# Plot: ULIRG→QSO sequence on a single νLν panel
# ============================================================================

fig, ax = plt.subplots(figsize=(8.0, 5.5))

# Color scheme: from deep red (ULIRG) to blue (QSO)
colors = ["#8B0000", "#DC143C", "#FF8C00", "#FFD700", "#4169E1"]
stages = [s["stage"] for s in sequence]

for i, (sed, color) in enumerate(zip(seds, colors)):
    ax.loglog(sed["wave"], sed["nu_l_nu"], color=color, lw=2.0, label=stages[i])

# Annotation: dust-dominated regime (FIR) vs. continuum-dominated regime (UV)
ax.axvline(100.0, color="gray", linestyle=":", alpha=0.4, linewidth=1.0)
ax.axvline(1e4, color="gray", linestyle=":", alpha=0.4, linewidth=1.0)
ax.text(50, 1e41, "UV", fontsize=9, color="gray", ha="right")
ax.text(5e3, 1e41, "FIR", fontsize=9, color="gray", ha="center")

ax.set_xlim(100, 1e6)
ax.set_ylim(1e38, 1e45)
ax.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]", fontsize=11)
ax.set_ylabel(r"$\nu L_\nu$ [erg s$^{-1}$]", fontsize=11)
ax.legend(loc="upper left", fontsize=9, framealpha=0.95)
ax.grid(True, which="both", alpha=0.2, linestyle="-", linewidth=0.5)

fig.tight_layout()
plt.savefig("plot_ulirg_to_qso_transition.png", dpi=150, bbox_inches="tight")